In [1]:
# Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"Device: {torch.cuda.get_device_name(0)}")

# Install Ultralytics YOLOv8
!pip install -q ultralytics opencv-python scikit-learn

# Clone the exact dataset repository
!git clone https://github.com/mvaldenegro/marine-debris-fls-datasets.git

CUDA Available: True
Device: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.5 MB/s eta 0:00:00
Cloning into 'marine-debris-fls-datasets'...
remote: Enumerating objects: 20438, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 20438 (delta 1), reused 0 (delta 0), pack-reused 20431 (from 2)
Receiving objects: 100% (20438/20438), 1.36 GiB | 19.83 MiB/s, done.
Resolving deltas: 100% (2171/2171), done.
Updating files: 100% (20131/20131), done.


In [2]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil
import yaml

# The 11 exact classes discovered in the dataset
CLASSES = [
    "Bottle", "Can", "Chain", "Drink-carton", "Hook", "Propeller",
    "Shampoo-bottle", "Standing-bottle", "Tire", "Valve", "Wall"
]
CLASS_MAP = {name: idx for idx, name in enumerate(CLASSES)}

# Paths
base_dir = Path("marine-debris-fls-datasets/md_fls_dataset/data/watertank-segmentation")
img_dir = base_dir / "Images"
ann_dir = base_dir / "BoxAnnotations"

# YOLO structure
yolo_dir = Path("/content/sonar_yolo_dataset")
for split in ['train', 'val']:
    (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

xml_files = list(ann_dir.glob("*.xml"))
train_xmls, val_xmls = train_test_split(xml_files, test_size=0.15, random_state=42)

def convert_to_yolo(xml_list, split_name):
    for xml_path in xml_list:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Image dimensions
        width = float(root.find('size/width').text)
        height = float(root.find('size/height').text)

        image_filename = root.find('filename').text
        image_path = img_dir / image_filename

        if not image_path.exists(): continue

        # Copy image
        shutil.copy(image_path, yolo_dir / 'images' / split_name / image_filename)

        # Parse labels
        label_file = yolo_dir / 'labels' / split_name / f"{image_path.stem}.txt"
        with open(label_file, 'w') as f:
            for obj in root.findall('object'):
                cls_name = obj.find('name').text
                if cls_name not in CLASS_MAP: continue
                cls_id = CLASS_MAP[cls_name]

                # Their XML format is x, y (top-left), w, h
                bbox = obj.find('bndbox')
                x = float(bbox.find('x').text)
                y = float(bbox.find('y').text)
                w = float(bbox.find('w').text)
                h = float(bbox.find('h').text)

                # Convert to YOLO format (normalized center_x, center_y, width, height)
                center_x = (x + w / 2) / width
                center_y = (y + h / 2) / height
                norm_w = w / width
                norm_h = h / height

                f.write(f"{cls_id} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")

convert_to_yolo(train_xmls, 'train')
convert_to_yolo(val_xmls, 'val')

# Generate dataset.yaml for YOLOv8
yaml_content = {
    'train': str(yolo_dir / 'images' / 'train'),
    'val': str(yolo_dir / 'images' / 'val'),
    'nc': len(CLASSES),
    'names': CLASSES
}
with open(yolo_dir / 'dataset.yaml', 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("Dataset successfully converted to YOLO format with Train/Val splits!")

Dataset successfully converted to YOLO format with Train/Val splits!


In [3]:
from ultralytics import YOLO

# Load YOLOv8 Medium (Best balance of speed and high accuracy for Colab T4)
model = YOLO('yolov8m.pt')

# Train the model with hyperparameters optimized for 90%+ Accuracy
results = model.train(
    data='/content/sonar_yolo_dataset/dataset.yaml',
    epochs=100,             # Increased epochs for higher accuracy
    imgsz=640,
    batch=16,
    device=0,
    patience=20,            # Early stopping if accuracy plateaus
    optimizer='auto',
    cos_lr=True,            # Cosine Learning rate for better convergence
    mosaic=1.0,             # Combines 4 images into 1 (Forces model to find tiny debris)
    mixup=0.1,              # Overlays images to prevent overfitting
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, # Grayscale/Sonar optimized hue shifts
    degrees=15.0,           # Rotations to simulate AUV pitch/roll
    flipud=0.5,             # Sonar can be flipped
    fliplr=0.5,
    project='sonar_debris_26057',
    name='yolov8m_high_acc'
)

print("Training Complete! Validating Model...")
# Run validation on the test set to print final accuracy (mAP)
metrics = model.val()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/sonar_yolo_dataset/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, f

In [4]:
from google.colab import files
import os

# Path to the best trained model weights
weight_path = "/content/sonar_debris_26057/yolov8m_high_acc/weights/best.pt"

if os.path.exists(weight_path):
    print("Downloading best.pt (High-Accuracy Model) to your computer...")
    files.download(weight_path)
else:
    print("Error: Weight file not found. Check training logs.")

Error: Weight file not found. Check training logs.


In [5]:
from google.colab import files
import os

# Corrected path based on your training logs
weight_path = "/content/runs/detect/sonar_debris_26057/yolov8m_high_acc/weights/best.pt"

if os.path.exists(weight_path):
    print("Found it! Downloading best.pt to your computer...")
    files.download(weight_path)
else:
    print("Still can't find it. Let's check where it is:")
    !find /content -name "best.pt"

Found it! Downloading best.pt to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>